# Lab 1

Build upon the bag-of-words and TF-IDF work you began during discussion - what can you do to make the model better? Explore the impact on the size of the vocabulary, data cleaning decisions, the size of the training set, or the type of classifier model used.

## Setup

In [1]:
# Import libraries
import numpy as np
import pandas as pd
from collections import Counter
from typing import List, Dict
import re

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB   
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Libraries loaded')

Libraries loaded


## Load IMDB Dataset

We'll use real movie reviews from IMDB.

In [2]:
print('Downloading IMDB dataset directly...\n')

import urllib.request
import tarfile
import os

# Download if not present
if not os.path.exists('aclImdb'):
    print('Downloading IMDB dataset (this may take a minute)...')
    url = 'https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz'
    urllib.request.urlretrieve(url, 'aclImdb_v1.tar.gz')
    
    # Extract
    with tarfile.open('aclImdb_v1.tar.gz', 'r:gz') as tar:
        tar.extractall()
    print('✓ Downloaded!')

# Load reviews from files
def load_imdb_data(path, num_samples=2500):
    reviews = []
    
    # Load positive reviews
    pos_path = os.path.join(path, 'train', 'pos')
    for i, filename in enumerate(os.listdir(pos_path)[:num_samples]):
        with open(os.path.join(pos_path, filename), 'r', encoding='utf-8') as f:
            reviews.append({'review': f.read(), 'sentiment': 'positive'})
    
    # Load negative reviews  
    neg_path = os.path.join(path, 'train', 'neg')
    for i, filename in enumerate(os.listdir(neg_path)[:num_samples]):
        with open(os.path.join(neg_path, filename), 'r', encoding='utf-8') as f:
            reviews.append({'review': f.read(), 'sentiment': 'negative'})
    
    return reviews

reviews_data = load_imdb_data('aclImdb', num_samples=2500)
data = pd.DataFrame(reviews_data)
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

print(f' Loaded {len(data)} reviews')


 Loaded 5000 reviews


## Exercise 1: Bag of Words 

### 1.1 Preprocessing

In [3]:
def preprocess_text(text: str) -> List[str]:
    """
    Clean and tokenize text
    """
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # TODO: Convert to lowercase and split
    tokens = text.lower().split() 

    # TODO: Remove punctuation
    cleaned = []
    for token in tokens:
        clean = re.sub(r'[^a-z0-9]', '', token)
        if clean:
            cleaned.append(clean)
    
    return cleaned



### 1.2 Build Vocabulary

In [4]:
def build_vocabulary(documents: List[str], max_features: int = 2000) -> Dict[str, int]:
    """
    Build vocabulary from documents
    """
    # TODO: Collect all words
    all_words = []
    for doc in documents:
        tokens = preprocess_text(doc)
        all_words.extend(tokens)
    
    # TODO: Get most common words
    word_counts = Counter(all_words) 
    # TODO: Create vocabulary dict
    most_common = word_counts.most_common(max_features)
    vocabulary = {word: i for i, (word, _) in enumerate(most_common)}
    
    return vocabulary

# Test
print('Building vocabulary...')
vocab = build_vocabulary(data['review'].tolist(), max_features=2000)
print(f'Vocabulary size: {len(vocab)}')



Building vocabulary...
Vocabulary size: 2000


### 1.3 Vectorize Documents

In [5]:
def vectorize_document(document: str, vocabulary: Dict[str, int]) -> np.ndarray:
    """
    Convert document to vector
    """
    # TODO: Initialize vector
    vector = np.zeros(len(vocabulary))
    
    # TODO: Count words
    tokens = preprocess_text(document)
    word_counts = Counter(tokens)
    
    # TODO: Fill vector
    for word, count in word_counts.items():
        if word in vocabulary:
            vector[vocabulary[word]] = count
    
    return vector

# Vectorize all reviews
print('Vectorizing reviews...')
X_bow = np.array([vectorize_document(r, vocab) for r in data['review']])
print(f'✓ Matrix shape: {X_bow.shape}')

Vectorizing reviews...
✓ Matrix shape: (5000, 2000)


## Exercise 2: TF-IDF 

### 2.1 Calculate TF

In [6]:
def calculate_tf(document: str) -> Dict[str, float]:
    """
    Calculate term frequency
    """
    tokens = preprocess_text(document)
    total = len(tokens)
    if total == 0:
        return {}
    
    # TODO: Calculate TF for each word
    counts = Counter(tokens)
    tf_scores = {w: c/total for w, c in counts.items()}
    
    return tf_scores

### 2.2 Calculate IDF

In [7]:
def calculate_idf(documents: List[str], vocabulary: Dict[str, int]) -> Dict[str, float]:
    """
    Calculate inverse document frequency
    """
    total_docs = len(documents)
    doc_count = {word: 0 for word in vocabulary}
    
    # TODO: Count documents containing each word
    for doc in documents:
        unique_words = set(preprocess_text(doc))
        for word in unique_words:
            if word in doc_count:
                doc_count[word] += 1
    
    # TODO: Calculate IDF
    idf_scores = {}
    for word, count in doc_count.items():
        idf_scores[word] = np.log(total_docs / (count + 1)) # +1 to avoid division by zero
    
    return idf_scores

print('Calculating IDF...')
idf_scores = calculate_idf(data['review'].tolist(), vocab)
print(' IDF calculated!')

Calculating IDF...
 IDF calculated!


### 2.3 Calculate TF-IDF Vectors

In [8]:
def calculate_tfidf_vector(document: str, vocabulary: Dict[str, int], idf_scores: Dict[str, float]) -> np.ndarray:
    """
    Calculate TF-IDF vector
    """
    vector = np.zeros(len(vocabulary))
    tf_scores = calculate_tf(document)
    
    # TODO: Calculate TF-IDF = TF × IDF
    for word, tf in tf_scores.items():
        if word in vocabulary:
            idx = vocabulary[word]
            idf = idf_scores.get(word, 0)
            vector[idx] = tf * idf    # <- replace pass
    
    return vector

print('Creating TF-IDF vectors...')
X_tfidf = np.array([calculate_tfidf_vector(r, vocab, idf_scores) for r in data['review']])
print(f'TF-IDF matrix shape: {X_tfidf.shape}')

Creating TF-IDF vectors...
TF-IDF matrix shape: (5000, 2000)


## Exercise 3: N-grams 

Quick intro to capturing phrases.

In [9]:
def extract_ngrams(text: str, n: int) -> List[str]:
    """
    Extract n-grams
    """
    tokens = preprocess_text(text)
    ngrams = []
    
    # TODO: Extract n-grams
    for i in range(len(tokens) - n + 1):
        ngrams.append(" ".join(tokens[i:i+n])) 
    
    return ngrams

# Test
print('Bigrams:', extract_ngrams('This movie was not good', 2))

Bigrams: ['this movie', 'movie was', 'was not', 'not good']


## Exercise 4: Build Classifier

### 4.1 Prepare Data

In [10]:
# Convert labels to numbers
y = (data['sentiment'] == 'positive').astype(int).values

# Split data
X_bow_train, X_bow_test, y_train, y_test = train_test_split(
    X_bow, y, test_size=0.2, random_state=42, stratify=y
)

X_tfidf_train, X_tfidf_test, _, _ = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training: {len(X_bow_train)}, Test: {len(X_bow_test)}')

Training: 4000, Test: 1000


### 4.2 Train with Bag of Words

In [11]:
# Train model
nb_bow = MultinomialNB()
nb_bow.fit(X_bow_train, y_train)

# Evaluate
y_pred_bow = nb_bow.predict(X_bow_test)
accuracy_bow = accuracy_score(y_test, y_pred_bow)

print(f'BoW Accuracy: {accuracy_bow:.4f} ({accuracy_bow*100:.1f}%)')
print('\n', classification_report(y_test, y_pred_bow, target_names=['Negative', 'Positive']))

BoW Accuracy: 0.8170 (81.7%)

               precision    recall  f1-score   support

    Negative       0.81      0.84      0.82       500
    Positive       0.83      0.80      0.81       500

    accuracy                           0.82      1000
   macro avg       0.82      0.82      0.82      1000
weighted avg       0.82      0.82      0.82      1000



### 4.3 Train with TF-IDF

In [12]:
# Train model
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_tfidf_train, y_train)

# Evaluate
y_pred_tfidf = nb_tfidf.predict(X_tfidf_test)
accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)

print(f'TF-IDF Accuracy: {accuracy_tfidf:.4f} ({accuracy_tfidf*100:.1f}%)')
print('\n', classification_report(y_test, y_pred_tfidf, target_names=['Negative', 'Positive']))

TF-IDF Accuracy: 0.8420 (84.2%)

               precision    recall  f1-score   support

    Negative       0.81      0.89      0.85       500
    Positive       0.88      0.80      0.83       500

    accuracy                           0.84      1000
   macro avg       0.84      0.84      0.84      1000
weighted avg       0.84      0.84      0.84      1000



### 4.4 Test on Your Reviews!

In [13]:
def predict_sentiment(review: str, model, use_tfidf=True):
    if use_tfidf:
        vector = calculate_tfidf_vector(review, vocab, idf_scores)
    else:
        vector = vectorize_document(review, vocab)
    vector = vector.reshape(1, -1)
    pred = model.predict(vector)[0]
    prob = model.predict_proba(vector)[0]
    return pred, prob

# Test reviews
test_reviews = [
    'This movie was absolutely incredible! Best film ever!',
    'Terrible waste of time. Boring and poorly made.',
    'Masterpiece! Brilliant acting and story!'
]

print('🎬 Testing Reviews:\n')
for review in test_reviews:
    pred, prob = predict_sentiment(review, nb_tfidf)
    sentiment = 'POSITIVE' if pred == 1 else 'NEGATIVE'
    conf = prob[pred] * 100
    print(f'{review}')
    print(f'  → {sentiment} ({conf:.1f}%)\n')

🎬 Testing Reviews:

This movie was absolutely incredible! Best film ever!
  → POSITIVE (58.6%)

Terrible waste of time. Boring and poorly made.
  → NEGATIVE (86.0%)

Masterpiece! Brilliant acting and story!
  → POSITIVE (70.2%)



###  Try Your Own!

Write a review of a movie you've seen:

In [14]:
# TODO: Write your movie review here!
my_review = 'I really enjoyed this movie. I thought the cinematography was great and the plot was interesting.'

pred, prob = predict_sentiment(my_review, nb_tfidf)
sentiment = ' POSITIVE' if pred == 1 else 'NEGATIVE'
print(f'Your review: {my_review}')
print(f'Prediction: {sentiment} ({prob[pred]*100:.1f}%)')

Your review: I really enjoyed this movie. I thought the cinematography was great and the plot was interesting.
Prediction:  POSITIVE (53.3%)


### 5.1 Improving our Baseline

In [37]:
# TODO: Let us try changing vocabulary size and see its effect on model performance
# Larger vocabulary should capture more words but it also may introduce more noise

print("\nBaseline vocabulary size: 2000")
print("Baseline TF-IDF + NB (vocab=2000):", accuracy_tfidf)

#Vocabulary sizes to test
for vocab_size in [1000, 2000, 3000, 5000, 10000, 20000, 50000, 100000]:
    print(f"\nVocabulary size: {vocab_size}")
    
    # Build vocabulary of the most frequent words
    vocab_tmp = build_vocabulary(data['review'].tolist(), max_features=vocab_size)

    # Compute IDF values for the current vocabulary
    idf_tmp = calculate_idf(data['review'].tolist(), vocab_tmp)
    
    # Convert all of our reviews into TF-IDF vectors
    X_tmp = np.array([
        calculate_tfidf_vector(r, vocab_tmp, idf_tmp)
        for r in data['review']
    ])
    
    # Split data into training and test data
    X_train, X_test, y_train, y_test = train_test_split(
        X_tmp, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Train model and evaluate
    model = MultinomialNB()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    
    print(f"TF-IDF + NB (vocab={vocab_size}) Accuracy:", accuracy)


Baseline vocabulary size: 2000
Baseline TF-IDF + NB (vocab=2000): 0.842

Vocabulary size: 1000
TF-IDF + NB (vocab=1000) Accuracy: 0.834

Vocabulary size: 2000
TF-IDF + NB (vocab=2000) Accuracy: 0.842

Vocabulary size: 3000
TF-IDF + NB (vocab=3000) Accuracy: 0.868

Vocabulary size: 5000
TF-IDF + NB (vocab=5000) Accuracy: 0.88

Vocabulary size: 10000
TF-IDF + NB (vocab=10000) Accuracy: 0.896

Vocabulary size: 20000
TF-IDF + NB (vocab=20000) Accuracy: 0.901

Vocabulary size: 50000
TF-IDF + NB (vocab=50000) Accuracy: 0.902

Vocabulary size: 100000
TF-IDF + NB (vocab=100000) Accuracy: 0.902


In [38]:
# TODO: Let us try data cleaning and removing stopwords and see its effect on model performance
STOPWORDS = {
    "the", "and", "is", "a", "to", "of", "in", "it", "that", "this",
    "for", "on", "with", "as", "was", "but", "are", "be"
}

def preprocess_text_stopwords(text):
    tokens = preprocess_text(text)
    return [t for t in tokens if t not in STOPWORDS]

print("\nBaseline vocabulary size (no stopword removal): 2000")
print("TF-IDF + NB Accuracy:", accuracy_tfidf)

# Vocabulary sizes to test
vocab_sizes = [1000, 2000, 3000, 5000, 10000, 20000, 50000, 100000]

for vocab_size in vocab_sizes:
    print(f"\nVocabulary size: {vocab_size}")
    
    # Preprocess all reviews with stopword removal
    clean_docs = [
        " ".join(preprocess_text_stopwords(r))
        for r in data["review"]
    ]
    
    # Build vocabulary and IDF on cleaned text
    vocab_sw = build_vocabulary(clean_docs, max_features=vocab_size)
    idf_sw = calculate_idf(clean_docs, vocab_sw)
    
    # Build our TF-IDF matrix
    X_sw = np.array([
        calculate_tfidf_vector(doc, vocab_sw, idf_sw)
        for doc in clean_docs
    ])
    
    # Split data into training and test data
    X_train, X_test, y_train, y_test = train_test_split(
        X_sw, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    # Train and evaluate model
    model = MultinomialNB()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, preds)
    print(f"TF-IDF + NB (stopwords removed, vocab={vocab_size}) Accuracy:", accuracy)


Baseline vocabulary size (no stopword removal): 2000
TF-IDF + NB Accuracy: 0.842

Vocabulary size: 1000
TF-IDF + NB (stopwords removed, vocab=1000) Accuracy: 0.836

Vocabulary size: 2000
TF-IDF + NB (stopwords removed, vocab=2000) Accuracy: 0.847

Vocabulary size: 3000
TF-IDF + NB (stopwords removed, vocab=3000) Accuracy: 0.869

Vocabulary size: 5000
TF-IDF + NB (stopwords removed, vocab=5000) Accuracy: 0.889

Vocabulary size: 10000
TF-IDF + NB (stopwords removed, vocab=10000) Accuracy: 0.904

Vocabulary size: 20000
TF-IDF + NB (stopwords removed, vocab=20000) Accuracy: 0.902

Vocabulary size: 50000
TF-IDF + NB (stopwords removed, vocab=50000) Accuracy: 0.905

Vocabulary size: 100000
TF-IDF + NB (stopwords removed, vocab=100000) Accuracy: 0.905


# 5.2 Observations/Analysis of Text Processing Exploration for Lab 1

I start from the TF-IDF Naive Bayes model first explored in our first discussion this week, using a vocabulary size of 2,000 as a baseline or reference point, and examine how performance changes as the feature space expands. For each specification, the TF-IDF representation is rebuilt while the classifier, preprocessing pipeline, and train–test split are held constant, so that changes in accuracy reflect vocabulary size rather than model choice, sampling variation or any other potential factors.

Expanding the vocabulary leads to meaningful gains in performance at first. Accuracy improves steadily as the model moves from very small vocabularies toward medium-sized ones, with particularly strong gains between 3,000 and 10,000 features. Beyond roughly 10,000–20,000 features, however, the improvement slows markedly. While the largest vocabularies achieve the highest observed accuracy, the incremental gains in performance at 50,000 features and above are small and unlikely to justify the additional complexity whether it be added computational costs, etc.

Introducing stopword removal shifts the accuracy curve upward but does not change its overall shape. Across most vocabulary sizes, removing common function words yields modest, but consistent improvements, especially at smaller and medium feature counts where very frequent terms otherwise dominate the representation. At larger vocabulary sizes, the advantage persists but narrows, suggesting that TF-IDF weighting already mitigates much of the noise introduced by high-frequency words. As shown in our outputs, there is a 0.003 performance improvement after stopword removal at the highest vocabulary size. 

Taken together, these results point to a clear trade-off. Most of the predictive signal for sentiment classification is captured once the vocabulary reaches a moderate scale which is around the 10,000-20,000 features mark, after which returns diminish quickly. Stopword removal provides a low-cost refinement that improves performance without altering the underlying model, while further expansion of the vocabulary delivers only marginal gains relative to the added dimensionality. There is one anomaly in our results which is the 20,000 for stopword removal drops in performance from 0.904 for 10,000 features to 0.902 for 20,000 features, and rises back to 0.905 for 50,000 features. I am unsure if this is because of something relating to the data, but it was a very interesting and confusing outlier in my predictions and expected results. 